# 03 — Projecting CCO to Schema.org

An organization's internal knowledge graph uses the Common Core
Ontologies (BFO-grounded, reified qualities, process/continuant
distinction).  The web team needs the same data as Schema.org for
structured data markup, search indexing, and API consumers.

These are not just different prefixes — they are genuinely different
modeling philosophies:

| Aspect | CCO | Schema.org |
|--------|-----|------------|
| Foundation | BFO (upper ontology) | Flat, web-pragmatic |
| Names | Reified via `InformationContentEntity` | Direct `schema:name` string |
| Roles | `ActOfOccupying` a `Role` | `schema:jobTitle` string |
| Locations | `GeospatialRegion` with lat/lon qualities | `schema:Place` with `schema:address` |
| Events | `ActOfOrganizing` (a BFO `Process`) | `schema:Event` (a thing with dates) |

The portal CONSTRUCT performs genuine structural transformation —
collapsing BFO reification into Schema.org's flat property model.

> Note: exact representations aligning with the latest CCO and Schema.org ontologies (class names, properties, etc.) is not w/in scope of the example. 

In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()

---
## Phase 1: CCO-Aligned Internal Knowledge Graph

The interior uses CCO patterns: persons are `cco:Person` (a BFO
`Object`), names are information content entities borne by
documents, affiliations connect persons to organizations via
roles, and events are processes with temporal regions.

For readability we use a CCO-aligned lightweight vocabulary rather
than the full CCO IRI scheme (which uses opaque OBO identifiers).
Each class carries a `skos:exactMatch` to the canonical CCO IRI.

In [ ]:
ds.add_holon("urn:holon:internal", "Internal Knowledge Graph")

ds.add_interior("urn:holon:internal", """
    @prefix cco:  <urn:cco:> .
    @prefix org:  <urn:internal:org:> .
    @prefix skos: <http://www.w3.org/2004/02/skos/core#> .

    # ── Person: Alice Chen ──────────────────────────────
    org:alice a cco:Person ;
        cco:hasFullName       "Alice Chen" ;
        cco:hasEmailAddress   "achen@example.org" ;
        cco:hasPhoneNumber    "+1-555-0142" ;
        cco:affiliatedWith    org:acme-corp ;
        cco:occupiesRole [
            a cco:Role ;
            cco:roleTitle     "Principal Ontologist" ;
            cco:roleStartDate "2022-03-15"^^xsd:date
        ] ;
        cco:hasExpertiseIn    org:topic-knowledge-graphs ,
                              org:topic-semantic-web ;
        cco:locatedAt         org:loc-vancouver ;
        .

    # ── Person: Bob Mwangi ──────────────────────────────
    org:bob a cco:Person ;
        cco:hasFullName       "Bob Mwangi" ;
        cco:hasEmailAddress   "bmwangi@example.org" ;
        cco:affiliatedWith    org:acme-corp ;
        cco:occupiesRole [
            a cco:Role ;
            cco:roleTitle     "Senior Data Engineer" ;
            cco:roleStartDate "2023-07-01"^^xsd:date
        ] ;
        cco:hasExpertiseIn    org:topic-data-engineering ,
                              org:topic-knowledge-graphs ;
        cco:locatedAt         org:loc-nairobi ;
        .

    # ── Organization: Acme Corp ─────────────────────────
    org:acme-corp a cco:Organization ;
        cco:hasOfficialName   "Acme Knowledge Systems" ;
        cco:foundedOnDate     "2015-06-12"^^xsd:date ;
        cco:hasWebsite        "https://acme-ks.example.org" ;
        cco:headquarteredAt   org:loc-vancouver ;
        cco:hasMember         org:alice ,
                              org:bob ;
        cco:operatesInDomain  org:topic-semantic-web ,
                              org:topic-knowledge-graphs ;
        .

    # ── Event: KG Conference 2026 ───────────────────────
    org:kgconf-2026 a cco:ActOfOrganizing ;
        cco:hasEventName      "Knowledge Graph Conference 2026" ;
        cco:hasEventAbbrev    "KGC-2026" ;
        cco:eventStartDate    "2026-05-04"^^xsd:date ;
        cco:eventEndDate      "2026-05-07"^^xsd:date ;
        cco:hasEventLocation  org:loc-new-york ;
        cco:hasParticipant    org:alice ,
                              org:bob ;
        cco:hasEventTopic     org:topic-knowledge-graphs ;
        .

    # ── Locations (GeospatialRegion) ────────────────────
    org:loc-vancouver a cco:GeospatialRegion ;
        cco:hasLocalityName   "Vancouver" ;
        cco:hasAdminRegion    "British Columbia" ;
        cco:hasCountryName    "Canada" ;
        cco:hasLatitude       49.2827 ;
        cco:hasLongitude      -123.1207 ;
        .

    org:loc-nairobi a cco:GeospatialRegion ;
        cco:hasLocalityName   "Nairobi" ;
        cco:hasCountryName    "Kenya" ;
        cco:hasLatitude       -1.2921 ;
        cco:hasLongitude      36.8219 ;
        .

    org:loc-new-york a cco:GeospatialRegion ;
        cco:hasLocalityName   "New York" ;
        cco:hasAdminRegion    "New York" ;
        cco:hasCountryName    "United States" ;
        cco:hasLatitude       40.7128 ;
        cco:hasLongitude      -74.0060 ;
        .

    # ── Topics / Expertise Areas ────────────────────────
    org:topic-knowledge-graphs a cco:InformationContentEntity ;
        cco:hasTextValue      "Knowledge Graphs" ;
        .

    org:topic-semantic-web a cco:InformationContentEntity ;
        cco:hasTextValue      "Semantic Web" ;
        .

    org:topic-data-engineering a cco:InformationContentEntity ;
        cco:hasTextValue      "Data Engineering" ;
        .
""")

ds.add_boundary("urn:holon:internal", """
    @prefix cco: <urn:cco:> .

    <urn:shapes:PersonShape> a sh:NodeShape ;
        sh:targetClass cco:Person ;
        sh:property [
            sh:path     cco:hasFullName ;
            sh:minCount 1 ;
            sh:maxCount 1 ;
            sh:datatype xsd:string ;
            sh:severity sh:Violation ;
            sh:message  "Person must have exactly one full name."
        ] ;
        sh:property [
            sh:path     cco:hasEmailAddress ;
            sh:minCount 1 ;
            sh:severity sh:Violation ;
            sh:message  "Person must have at least one email."
        ] ;
        sh:property [
            sh:path     cco:affiliatedWith ;
            sh:minCount 1 ;
            sh:nodeKind sh:IRI ;
            sh:severity sh:Warning ;
            sh:message  "Person should have an organizational affiliation."
        ] ;
        .

    <urn:shapes:OrganizationShape> a sh:NodeShape ;
        sh:targetClass cco:Organization ;
        sh:property [
            sh:path     cco:hasOfficialName ;
            sh:minCount 1 ;
            sh:maxCount 1 ;
            sh:datatype xsd:string ;
            sh:severity sh:Violation ;
            sh:message  "Organization must have one official name."
        ] ;
        .

    <urn:shapes:EventShape> a sh:NodeShape ;
        sh:targetClass cco:ActOfOrganizing ;
        sh:property [
            sh:path     cco:hasEventName ;
            sh:minCount 1 ;
            sh:severity sh:Violation
        ] ;
        sh:property [
            sh:path     cco:eventStartDate ;
            sh:minCount 1 ;
            sh:datatype xsd:date ;
            sh:severity sh:Violation
        ] ;
        .
""")

# Validate source membrane
result = ds.validate_membrane("urn:holon:internal")
print(result.summary())

---
## Phase 2: Schema.org Target Holon

The boundary defines what valid Schema.org output looks like.
This IS the API contract — any portal targeting this holon must
produce data that conforms to these shapes.

In [ ]:
ds.add_holon("urn:holon:schemaorg", "Schema.org Web Data")

ds.add_boundary("urn:holon:schemaorg", """
    @prefix schema: <https://schema.org/> .

    <urn:shapes:SchemaPersonShape> a sh:NodeShape ;
        sh:targetClass schema:Person ;
        sh:property [
            sh:path     schema:name ;
            sh:minCount 1 ;
            sh:maxCount 1 ;
            sh:datatype xsd:string ;
            sh:severity sh:Violation ;
            sh:message  "schema:Person requires exactly one name."
        ] ;
        sh:property [
            sh:path     schema:email ;
            sh:minCount 1 ;
            sh:severity sh:Violation ;
            sh:message  "schema:Person requires at least one email."
        ] ;
        sh:property [
            sh:path     schema:affiliation ;
            sh:nodeKind sh:IRI ;
            sh:severity sh:Warning
        ] ;
        sh:property [
            sh:path     schema:jobTitle ;
            sh:maxCount 1 ;
            sh:datatype xsd:string ;
            sh:severity sh:Info
        ] ;
        .

    <urn:shapes:SchemaOrgShape> a sh:NodeShape ;
        sh:targetClass schema:Organization ;
        sh:property [
            sh:path     schema:name ;
            sh:minCount 1 ;
            sh:maxCount 1 ;
            sh:datatype xsd:string ;
            sh:severity sh:Violation ;
            sh:message  "schema:Organization requires exactly one name."
        ] ;
        sh:property [
            sh:path     schema:url ;
            sh:maxCount 1 ;
            sh:severity sh:Warning
        ] ;
        sh:property [
            sh:path     schema:foundingDate ;
            sh:maxCount 1 ;
            sh:datatype xsd:date ;
            sh:severity sh:Info
        ] ;
        .

    <urn:shapes:SchemaEventShape> a sh:NodeShape ;
        sh:targetClass schema:Event ;
        sh:property [
            sh:path     schema:name ;
            sh:minCount 1 ;
            sh:severity sh:Violation
        ] ;
        sh:property [
            sh:path     schema:startDate ;
            sh:minCount 1 ;
            sh:datatype xsd:date ;
            sh:severity sh:Violation ;
            sh:message  "schema:Event requires a start date."
        ] ;
        sh:property [
            sh:path     schema:endDate ;
            sh:maxCount 1 ;
            sh:datatype xsd:date ;
            sh:severity sh:Info
        ] ;
        sh:property [
            sh:path     schema:location ;
            sh:nodeKind sh:IRI ;
            sh:severity sh:Warning
        ] ;
        .

    <urn:shapes:SchemaPlaceShape> a sh:NodeShape ;
        sh:targetClass schema:Place ;
        sh:property [
            sh:path     schema:name ;
            sh:minCount 1 ;
            sh:severity sh:Violation
        ] ;
        sh:property [
            sh:path     schema:latitude ;
            sh:maxCount 1 ;
            sh:datatype xsd:decimal ;
            sh:severity sh:Warning
        ] ;
        sh:property [
            sh:path     schema:longitude ;
            sh:maxCount 1 ;
            sh:datatype xsd:decimal ;
            sh:severity sh:Warning
        ] ;
        .
""")

print(ds.summary())

---
## Phase 3: Alignment Axioms

Property-level mappings between CCO and Schema.org.  These are
genuine structural transformations, not just prefix swaps:

- CCO reifies roles as blank-node `Role` entities with a `roleTitle`.
  Schema.org flattens this to a `schema:jobTitle` string on the person.
- CCO separates locality, admin region, and country as distinct
  properties on a `GeospatialRegion`.  Schema.org uses `schema:address`.
- CCO models events as `ActOfOrganizing` (a BFO Process).
  Schema.org models events as `schema:Event` (a thing with dates).

In [ ]:
ds.add_holon("urn:holon:alignment:cco-schema", "CCO ↔ Schema.org Alignment")

ds.add_interior("urn:holon:alignment:cco-schema", """
    @prefix cco:    <urn:cco:> .
    @prefix schema: <https://schema.org/> .

    # ── Class alignment ─────────────────────────────────
    cco:Person           rdfs:subClassOf schema:Person .
    cco:Organization     rdfs:subClassOf schema:Organization .
    cco:ActOfOrganizing  rdfs:subClassOf schema:Event .
    cco:GeospatialRegion rdfs:subClassOf schema:Place .

    # ── Property alignment ──────────────────────────────
    #
    # Direct mappings (same modeling pattern, different names):
    cco:hasFullName       rdfs:subPropertyOf schema:name .
    cco:hasOfficialName   rdfs:subPropertyOf schema:name .
    cco:hasEventName      rdfs:subPropertyOf schema:name .
    cco:hasLocalityName   rdfs:subPropertyOf schema:name .
    cco:hasEmailAddress   rdfs:subPropertyOf schema:email .
    cco:hasPhoneNumber    rdfs:subPropertyOf schema:telephone .
    cco:hasWebsite        rdfs:subPropertyOf schema:url .
    cco:affiliatedWith    rdfs:subPropertyOf schema:affiliation .
    cco:hasParticipant    rdfs:subPropertyOf schema:attendee .
    cco:hasLatitude       rdfs:subPropertyOf schema:latitude .
    cco:hasLongitude      rdfs:subPropertyOf schema:longitude .
    cco:foundedOnDate     rdfs:subPropertyOf schema:foundingDate .
    cco:eventStartDate    rdfs:subPropertyOf schema:startDate .
    cco:eventEndDate      rdfs:subPropertyOf schema:endDate .
    cco:hasEventLocation  rdfs:subPropertyOf schema:location .
    cco:hasMember         rdfs:subPropertyOf schema:member .
    cco:headquarteredAt   rdfs:subPropertyOf schema:location .
""")

---
## Phase 4: The Portal

This CONSTRUCT does the heavy lifting.  It transforms CCO's
reified, BFO-grounded structure into Schema.org's flat model.

Key structural transformations:
- `cco:occupiesRole [ cco:roleTitle ?t ]` → `schema:jobTitle ?t`
  (blank-node role entity collapsed to a direct string property)
- `cco:ActOfOrganizing` → `schema:Event`
  (BFO Process retyped as a Schema.org thing)
- `cco:GeospatialRegion` → `schema:Place`
  (administrative decomposition → flat place with coordinates)
- `cco:hasExpertiseIn [ cco:hasTextValue ?v ]` → `schema:knowsAbout ?v`
  (information content entity collapsed to direct string)

In [ ]:
ds.add_portal(
    "urn:portal:cco-to-schemaorg",
    "urn:holon:internal",
    "urn:holon:schemaorg",
    label="CCO → Schema.org",
    construct_query="""
        PREFIX cco:    <urn:cco:>
        PREFIX schema: <https://schema.org/>
        PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>

        CONSTRUCT {
            # ── Persons ──
            ?person a schema:Person ;
                schema:name         ?personName ;
                schema:email        ?email ;
                schema:telephone    ?phone ;
                schema:affiliation  ?org ;
                schema:jobTitle     ?roleTitle ;
                schema:knowsAbout   ?expertiseLabel ;
                schema:homeLocation ?personLoc ;
                .

            # ── Organizations ──
            ?org a schema:Organization ;
                schema:name         ?orgName ;
                schema:url          ?website ;
                schema:foundingDate ?founded ;
                schema:location     ?hqLoc ;
                schema:member       ?member ;
                .

            # ── Events ──
            ?event a schema:Event ;
                schema:name      ?eventName ;
                schema:startDate ?startDate ;
                schema:endDate   ?endDate ;
                schema:location  ?eventLoc ;
                schema:attendee  ?attendee ;
                .

            # ── Places ──
            ?place a schema:Place ;
                schema:name      ?localityName ;
                schema:latitude  ?lat ;
                schema:longitude ?lon ;
                .
        }
        WHERE {
            GRAPH <urn:holon:internal/interior> {
                # ── Persons ──
                {
                    ?person a cco:Person ;
                        cco:hasFullName     ?personName ;
                        cco:hasEmailAddress ?email ;
                        .
                    OPTIONAL { ?person cco:hasPhoneNumber  ?phone }
                    OPTIONAL { ?person cco:affiliatedWith  ?org }
                    OPTIONAL { ?person cco:locatedAt       ?personLoc }
                    OPTIONAL {
                        ?person cco:occupiesRole ?role .
                        ?role cco:roleTitle ?roleTitle
                    }
                    OPTIONAL {
                        ?person cco:hasExpertiseIn ?expertise .
                        ?expertise cco:hasTextValue ?expertiseLabel
                    }
                }
                UNION
                # ── Organizations ──
                {
                    ?org a cco:Organization ;
                        cco:hasOfficialName ?orgName ;
                        .
                    OPTIONAL { ?org cco:hasWebsite       ?website }
                    OPTIONAL { ?org cco:foundedOnDate    ?founded }
                    OPTIONAL { ?org cco:headquarteredAt  ?hqLoc }
                    OPTIONAL { ?org cco:hasMember        ?member }
                }
                UNION
                # ── Events ──
                {
                    ?event a cco:ActOfOrganizing ;
                        cco:hasEventName   ?eventName ;
                        cco:eventStartDate ?startDate ;
                        .
                    OPTIONAL { ?event cco:eventEndDate     ?endDate }
                    OPTIONAL { ?event cco:hasEventLocation ?eventLoc }
                    OPTIONAL { ?event cco:hasParticipant   ?attendee }
                }
                UNION
                # ── Places ──
                {
                    ?place a cco:GeospatialRegion ;
                        cco:hasLocalityName ?localityName ;
                        .
                    OPTIONAL { ?place cco:hasLatitude  ?lat }
                    OPTIONAL { ?place cco:hasLongitude ?lon }
                }
            }
        }
    """,
)

print(ds.summary())

---
## Phase 5: Traverse, Validate, Record Provenance

In [ ]:
projected, membrane = ds.traverse(
    "urn:holon:internal",
    "urn:holon:schemaorg",
    inject=True,
    validate=True,
    agent_iri="urn:agent:web-publisher-v1",
)

print(f"Projected {len(projected)} triples\n")
print(membrane.summary())

---
## Phase 6: Inspect the Schema.org Output

Query the target holon's interior — this is now valid Schema.org
data, ready for JSON-LD embedding or API serving.

In [ ]:
print("── schema:Person ──")
for r in ds.query("""
    PREFIX schema: <https://schema.org/>
    SELECT ?person ?name ?email ?title ?affil
    WHERE {
        GRAPH <urn:holon:schemaorg/interior> {
            ?person a schema:Person ;
                schema:name  ?name ;
                schema:email ?email .
            OPTIONAL { ?person schema:jobTitle    ?title }
            OPTIONAL { ?person schema:affiliation ?affil }
        }
    }
    ORDER BY ?name
"""):
    affil = r.get("affil", "")
    affil_short = affil.rsplit(":", 1)[-1] if affil else "—"
    print(f"  {r['name']:20s}  {r['email']:28s}  "
          f"{r.get('title', '—'):25s}  @{affil_short}")

In [ ]:
print("\n── schema:Organization ──")
for r in ds.query("""
    PREFIX schema: <https://schema.org/>
    SELECT ?org ?name ?url ?founded
    WHERE {
        GRAPH <urn:holon:schemaorg/interior> {
            ?org a schema:Organization ;
                schema:name ?name .
            OPTIONAL { ?org schema:url          ?url }
            OPTIONAL { ?org schema:foundingDate ?founded }
        }
    }
"""):
    print(f"  {r['name']:30s}  {r.get('url', '—'):35s}  "
          f"founded: {r.get('founded', '—')}")

In [ ]:
print("\n── schema:Event ──")
for r in ds.query("""
    PREFIX schema: <https://schema.org/>
    SELECT ?event ?name ?start ?end ?loc
    WHERE {
        GRAPH <urn:holon:schemaorg/interior> {
            ?event a schema:Event ;
                schema:name      ?name ;
                schema:startDate ?start .
            OPTIONAL { ?event schema:endDate  ?end }
            OPTIONAL { ?event schema:location ?loc }
        }
    }
"""):
    loc = r.get("loc", "")
    loc_short = loc.rsplit(":", 1)[-1] if loc else "—"
    print(f"  {r['name']:40s}  {r['start']} → {r.get('end', '—')}"
          f"  @{loc_short}")

In [ ]:
print("\n── schema:Place ──")
for r in ds.query("""
    PREFIX schema: <https://schema.org/>
    SELECT ?place ?name ?lat ?lon
    WHERE {
        GRAPH <urn:holon:schemaorg/interior> {
            ?place a schema:Place ;
                schema:name ?name .
            OPTIONAL { ?place schema:latitude  ?lat }
            OPTIONAL { ?place schema:longitude ?lon }
        }
    }
    ORDER BY ?name
"""):
    print(f"  {r['name']:20s}  ({r.get('lat', '?')}, {r.get('lon', '?')})")

---
## Phase 7: What the Portal Transformed

The structural changes are significant — this is not a rename:

| CCO Pattern | Schema.org Result |
|-------------|-------------------|
| `cco:occupiesRole [ cco:roleTitle "Principal Ontologist" ]` | `schema:jobTitle "Principal Ontologist"` |
| `cco:hasExpertiseIn [ cco:hasTextValue "Knowledge Graphs" ]` | `schema:knowsAbout "Knowledge Graphs"` |
| `cco:ActOfOrganizing` (BFO Process) | `schema:Event` |
| `cco:GeospatialRegion` with decomposed admin fields | `schema:Place` with `schema:name` |
| `cco:hasMember` (org → person) | `schema:member` |
| `cco:hasParticipant` (event → person) | `schema:attendee` |

The blank-node `cco:Role` was collapsed: the portal's CONSTRUCT
navigated `?person cco:occupiesRole ?role . ?role cco:roleTitle ?t`
and projected the flat `schema:jobTitle ?t`.  The intermediate
blank node does not appear in the Schema.org output.

Same for expertise: the `cco:InformationContentEntity` with its
`cco:hasTextValue` was collapsed to a direct `schema:knowsAbout`
string.  The ICE node is gone.

---
## Phase 8: Provenance Trail

In [ ]:
g = ds.backend.get_graph("urn:holon:schemaorg/context")
print(f"Provenance ({len(g)} triples):\n")
for s, p, o in sorted(g):
    s_short = str(s).rsplit(":", 1)[-1][:40]
    p_short = str(p).rsplit("/", 1)[-1].rsplit("#", 1)[-1]
    o_short = str(o)[:55]
    print(f"  {s_short:42s} {p_short:28s} {o_short}")

---
## Phase 9: Semantic Equivalence

Both holons exist in the same holarchy.  The alignment axioms
declare `rdfs:subPropertyOf` relationships — so after RDFS
entailment, data from either vocabulary is queryable via the
superproperty.

In [ ]:
print("Alignment axioms (property mappings):\n")
for r in ds.query("""
    SELECT ?cco_prop ?schema_prop
    WHERE {
        GRAPH <urn:holon:alignment:cco-schema/interior> {
            ?cco_prop rdfs:subPropertyOf ?schema_prop .
        }
    }
    ORDER BY ?cco_prop
"""):
    cco = r["cco_prop"].rsplit(":", 1)[-1]
    sch = r["schema_prop"].rsplit("/", 1)[-1]
    print(f"  {cco:25s}  ⊑  schema:{sch}")

---
## Summary

```
┌─────────────────────────────────────────────────────────────┐
│  CCO Interior                  Schema.org Interior          │
│  ┌───────────────────┐         ┌───────────────────┐       │
│  │ cco:Person        │ portal  │ schema:Person     │       │
│  │  hasFullName      │───────→ │  name             │       │
│  │  occupiesRole [   │         │  jobTitle          │       │
│  │    roleTitle      │         │  email             │       │
│  │  ]                │         │  affiliation       │       │
│  │  hasExpertiseIn [ │         │  knowsAbout        │       │
│  │    hasTextValue   │         │                    │       │
│  │  ]                │         │                    │       │
│  ├───────────────────┤         ├───────────────────┤       │
│  │ cco:Organization  │───────→ │ schema:Org        │       │
│  ├───────────────────┤         ├───────────────────┤       │
│  │ cco:ActOfOrganiz  │───────→ │ schema:Event      │       │
│  ├───────────────────┤         ├───────────────────┤       │
│  │ cco:GeospatialReg │───────→ │ schema:Place      │       │
│  └───────────────────┘         └───────────────────┘       │
│                                                             │
│  The portal CONSTRUCT collapses BFO reification:            │
│    - Blank-node Role → flat jobTitle                        │
│    - ICE with hasTextValue → flat knowsAbout                │
│    - BFO Process → Schema.org thing                         │
│    - Admin decomposition → flat Place                       │
│                                                             │
│  The alignment holon holds rdfs:subPropertyOf axioms.       │
│  The target boundary SHACL shapes ARE the API contract.     │
│  Provenance is recorded as PROV-O in the context graph.     │
└─────────────────────────────────────────────────────────────┘
```